In [ ]:
# Importy bibliotek
import torch
from torchvision import datasets, transforms, models
import numpy as np
from torchvision.models import VGG
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from torch.utils.data import DataLoader, Subset
from preprocess import get_transforms
from config import DATA_PATH

# **Przygotowanie danych i transformacje**

In [ ]:
# Domyślny transform — zostanie nadpisany przez preprocess.get_transforms() w main()
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform)

print(f'Pomyslnie wczytano obrazow: {len(dataset)}')
print(f'Zidentyfikowane klasy: {dataset.classes}')
print(f'Slownik klas (etykiety numeryczne): {dataset.class_to_idx}')

# **Inicjalizacja modeli**

In [ ]:
def compute_mean_std(data_path):
    base_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
    ])
    ds = datasets.ImageFolder(root=data_path, transform=base_transform)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)

    mean = torch.zeros(3)
    std = torch.zeros(3)
    n_samples = 0

    for images, _ in loader:
        batch_size = images.size(0)
        images = images.view(batch_size, 3, -1)
        mean += images.mean(dim=2).sum(dim=0)
        std += images.std(dim=2).sum(dim=0)
        n_samples += batch_size

    mean /= n_samples
    std /= n_samples

    print(f"Mean: {mean.tolist()}")
    print(f"Std:  {std.tolist()}")
    return mean.tolist(), std.tolist()

dataset_mean, dataset_std = compute_mean_std(data_path)

In [ ]:
def get_model(model_name):
  numer_of_classes = 7 # 7 klas: 'angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised'

  if model_name == 'vgg':
    model = models.vgg16(weights=models.VGG16_Weights)
    for param in model.parameters():
      param.requires_grad = False
    model.classifier[6] = torch.nn.Linear(4096, numer_of_classes)
  elif model_name == 'resnet18':
    model = models.resnet18(weights=models.ResNet18_Weights)
    for param in model.parameters():
      param.requires_grad = False
    model.fc = torch.nn.Linear(512, numer_of_classes)
  elif model_name == 'resnet50':
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    for param in model.parameters():
      param.requires_grad = False
    model.fc = torch.nn.Linear(2048, numer_of_classes)
  else:
    raise ValueError("Nieznany model")

  return model.to(device)

In [ ]:
def train_and_evaluate(model_name, dataset, test_dataset, class_weights):
  criterion = nn.CrossEntropyLoss(weight=class_weights)
  optimizer = optim.Adam(model_name.parameters(), lr=0.001)

  model_name.train()
  for inputs, labels in dataset:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()
    output = model_name(inputs)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

  model_name.eval()
  all_predictions = []
  correct_predictions = []
  with torch.no_grad():
      for inputs, labels in test_dataset:
          inputs, labels = inputs.to(device), labels.to(device)
          outputs = model_name(inputs)
          _, batch_preds = torch.max(outputs.data, 1)
          all_predictions.extend(batch_preds.cpu().numpy())
          correct_predictions.extend(labels.cpu().numpy())

  acc = accuracy_score(correct_predictions, all_predictions)
  bal_acc = balanced_accuracy_score(correct_predictions, all_predictions)
  prec = precision_score(correct_predictions, all_predictions, average='weighted', zero_division=0)
  rec = recall_score(correct_predictions, all_predictions, average='weighted', zero_division=0)
  cm = confusion_matrix(correct_predictions, all_predictions)

  return acc, bal_acc, prec, rec, cm

In [ ]:
def run_experiment(model_name, dataset):
  num_classes = len(dataset.classes)
  accuracies = []
  balanced_accuracies = []
  precisions = []
  recalls = []
  confusion_matrices = []

  rskf = RepeatedStratifiedKFold(n_splits=2, n_repeats=5, random_state=42)

  for fold, (train_index, test_index) in enumerate(rskf.split(np.arange(len(dataset)), dataset.targets)):

    train_subset = Subset(dataset, train_index)
    test_subset = Subset(dataset, test_index)

    # Obliczenie wag klas z podzbioru treningowego do CrossEntropy
    train_targets = torch.tensor([dataset.targets[i] for i in train_index])
    class_counts = torch.bincount(train_targets, minlength=num_classes).float()
    class_weights = len(train_index) / (num_classes * class_counts)
    class_weights = class_weights.to(device)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=32, shuffle=False)

    model = get_model(model_name)
    acc, bal_acc, prec, rec, cm = train_and_evaluate(model, train_loader, test_loader, class_weights)

    accuracies.append(acc)
    balanced_accuracies.append(bal_acc)
    precisions.append(prec)
    recalls.append(rec)
    confusion_matrices.append(cm)

    print(f"Accuracy: {acc:.4f}, Balanced Accuracy: {bal_acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")

  return accuracies, balanced_accuracies, precisions, recalls, confusion_matrices

In [ ]:
def main(num_models, model_names, mean=None, std=None):
    if len(model_names) != num_models:
        raise ValueError(f'Podano num_models={num_models}, ale lista zawiera {len(model_names)} modeli.')

    results = {}
    for model_name in model_names:
        print(f'\n=== Trening modelu: {model_name} ===')
        transform = get_transforms(model_name, mean=mean, std=std)
        model_dataset = datasets.ImageFolder(root=DATA_PATH, transform=transform)
        accs, bal_accs, precs, recs, cms = run_experiment(model_name, model_dataset)
        results[model_name] = {
            'accuracies':          accs,
            'balanced_accuracies': bal_accs,
            'precisions':          precs,
            'recalls':             recs,
            'confusion_matrices':  cms,
        }
        print(f'Srednia Accuracy:          {sum(accs)/len(accs):.4f}')
        print(f'Srednia Balanced Accuracy: {sum(bal_accs)/len(bal_accs):.4f}')
        print(f'Srednia Precision:         {sum(precs)/len(precs):.4f}')
        print(f'Srednia Recall:            {sum(recs)/len(recs):.4f}')

    return results

In [ ]:
results = main(
    num_models=3,
    model_names=['resnet18', 'vgg', 'resnet50'],
    mean=dataset_mean, # Custom parametry jesli wlasny dataset
    std=dataset_std,
)

In [ ]:
def plot_confusion_matrices(results, class_names):
    n_models = len(results)
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
    if n_models == 1:
        axes = [axes]

    for ax, (model_name, metrics) in zip(axes, results.items()):
        mean_cm = np.mean(metrics['confusion_matrices'], axis=0)
        mean_cm_normalized = mean_cm / mean_cm.sum(axis=1, keepdims=True)

        sns.heatmap(
            mean_cm_normalized,
            annot=True,
            fmt='.2f',
            cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            vmin=0, vmax=1,
            ax=ax
        )
        bal_acc_mean = np.mean(metrics['balanced_accuracies'])
        ax.set_title(f'Macierz pomylek: {model_name}\nSrednia Balanced Accuracy: {bal_acc_mean:.4f}')
        ax.set_xlabel('Predykcja')
        ax.set_ylabel('Rzeczywista klasa')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    plt.tight_layout()
    plt.show()

class_names = dataset.classes
plot_confusion_matrices(results, class_names)